# Forme canonique commandable

In [1]:
import numpy as np
from numpy.linalg import inv

np.set_printoptions(suppress=True)

L'équation d'évolution est sous la forme $\dfrac{dx}{dt} = Ax + Bu$

In [2]:
A = np.array([[0, 1, 0], [100, 0, 2], [0, 0, -100]])
A

array([[   0,    1,    0],
       [ 100,    0,    2],
       [   0,    0, -100]])

In [3]:
B = np.array([[0], [0], [100]])
B

array([[  0],
       [  0],
       [100]])

Calcul du polynôme caractéristique pour le système initial

In [4]:
Pcarac = np.poly(A)
print(Pcarac)

[     1.    100.   -100. -10000.]


Polynôme caratéristique souhaité pour le système avec retour d'état $\dfrac{dx}{dt} = (A-BK)x + Be$

In [5]:
Pcible = np.array([1, 60, 600, 5000])
print(Pcible)

[   1   60  600 5000]


Calcul de la matrice $V_G$

In [6]:
n = A.shape[0]
VG = np.zeros((n, n))
for k in range(n):
    VG[k, k:n] = Pcarac[0:n-k]
print(VG)

[[   1.  100. -100.]
 [   0.    1.  100.]
 [   0.    0.    1.]]


Calcul de la matrice $Q_G$

In [7]:
QG = B
for k in range(0, n-1):
    colonne = np.reshape(QG[:, k], (n, 1))
    QG = np.concatenate((QG, A@colonne), axis=1)
print(QG)

[[      0       0     200]
 [      0     200  -20000]
 [    100  -10000 1000000]]


On vérifie que la matrice $\overline{\overline A} = V_G^{-1}Q_G^{-1}AQ_G V_G$ a bien la forme requise

In [8]:
VGinv = inv(VG)
QGinv = inv(QG)
Abarre = VGinv@QGinv@A@QG@VG
print(Abarre)

[[ -100.   100. 10000.]
 [    1.     0.     0.]
 [    0.     1.     0.]]


On vérifie que le vecteur $\overline{\overline B} = V_G^{-1}Q_G^{-1}B$ a bien la forme requise

In [9]:
Bbarre = VGinv@QGinv@B
print(Bbarre)

[[1.]
 [0.]
 [0.]]


Dans la nouvelle base, le vecteur ligne $\overline{\overline K}$ est simplement la différence entre les coefficients du polynôme cible et les coefficients du polynôme caractéristique initial

In [10]:
Kbarre = Pcible[1:]-Pcarac[1:]
print(Kbarre)

[  -40.   700. 15000.]


Le retour à la base initiale donne le vecteur $K=\overline{\overline K}V_G^{-1}Q_G^{-1}$ pour le retour d'état avec une commande $u = e - Kx$.

In [11]:
K = Kbarre@VGinv@QGinv
print(K)

[55.   3.5 -0.4]


La fonction qui reprend les étapes du calcul, utilisable dans le cas général, sous réserve que la matrice $Q_G$ soit de rang plein.

In [12]:
def commande(A, B, Pcible):
    n = A.shape[0]
    Pcarac = np.poly(A)
    VG = np.zeros((n, n))
    for k in range(n):
        VG[k, k:n] = Pcarac[0:n-k]
    QG = B
    for k in range(0, n-1):
        colonne = np.reshape(QG[:, k], (n, 1))
        QG = np.concatenate((QG, A@colonne), axis=1)
    K = (Pcible[1:]-Pcarac[1:])@inv(VG)@inv(QG)
    return K

In [13]:
print(commande(A, B, Pcible))

[55.   3.5 -0.4]
